# Simple linear regression with no tuning

In [23]:
import numpy as np
import pandas as pd
import sys

from pathlib import Path
from sklearn.linear_model import LogisticRegression

# Own functions
# Necessary to import from src dir
sys.path.append("..")
import src.kaggle_helpers as kh
import src.preprocessing_minimal as ppm

## Reading in the data

In [24]:
from sklearn.model_selection import train_test_split

data_dir = Path('data')
output_dir = Path('outputs')

train_df = ppm.preprocess_train_data(data_dir / 'train.csv',drop_std=True)
final_test_df = pd.read_csv(data_dir / 'test.csv').drop(columns=["partlybad", "date"])
final_test_df = final_test_df.drop(columns=[col for col in final_test_df.columns if col.endswith('.std')])


train_df, test_df = train_test_split(train_df, test_size=0.3, random_state=42)

#train_df = ppm.generate_synthetic_data(train_df, num_datasets=100)

## Splitting to x and y variables

In [25]:
X_train, y_train2, y_train4 = ppm.split_xy(train_df)
X_test, y_test2, y_test4 = ppm.split_xy(test_df)
#X_test = X_test.drop(columns=[col for col in X_test.columns if col.endswith('.std')])

## Constructing event-nonevent classifier

In [26]:
#Fitting logistic regression to class2
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)

log_model = LogisticRegression(C=0.8, l1_ratio=1, solver='liblinear')
log_model.fit(X_train_scaled, y_train2)

from sklearn.metrics import classification_report
X_test_scaled = scaler.transform(X_test)
y_pred = log_model.predict(X_test_scaled)
print(classification_report(y_test2, y_pred))


              precision    recall  f1-score   support

       event       0.85      0.89      0.87        70
    nonevent       0.87      0.83      0.85        65

    accuracy                           0.86       135
   macro avg       0.86      0.86      0.86       135
weighted avg       0.86      0.86      0.86       135



## Constructing extremely simple random forest class4 classifier (not a good method)

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train4)
print(classification_report(y_test4, rf.predict(X_test_scaled)))


              precision    recall  f1-score   support

          II       0.40      0.49      0.44        35
          Ia       0.50      0.12      0.20         8
          Ib       0.44      0.30      0.36        27
    nonevent       0.79      0.88      0.83        65

    accuracy                           0.61       135
   macro avg       0.53      0.45      0.46       135
weighted avg       0.60      0.61      0.60       135



### Calculating a estimated Kaggle score

In [28]:
total_score = kh.simulate_kaggle_scoring(X_test_scaled, y_test2, y_test4, log_model, log_model, rf, print_partial_scores=True)
print(f'Kaggle score: {total_score:.4f}')

Binary accuracy: 0.8593
Multiclass accuracy: 0.6148
Model perplexity: 1.3695
Kaggle score: 0.7015


### Forming the output df

In [29]:
out_df = kh.generate_kaggle_output(
    final_test_df,
    model_prob=log_model,
    model_class4=rf,
    scaler=scaler
)
out_df.to_csv(output_dir / 'not_good_submission.csv', index=False)
out_df.head(10)

,id,class4,p
0,450,nonevent,0.646176
1,451,Ib,0.979923
2,452,nonevent,0.011834
3,453,nonevent,0.287010
4,454,Ib,0.874244
5,455,nonevent,0.112729
6,456,Ib,0.948179
7,457,Ib,0.974009
8,458,nonevent,0.223621
9,459,nonevent,0.172432
